# 05 E5 blogger embedding worker

Encode one exact job artifact in the isolated 768-dimensional E5 space.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import os
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = '9cfe2210c0f6d689445aadf917f1638dcaa173178e36f48832e91bfdeeafebfd'
RUNTIME_CONTRACT = 'my-data-hub-blogger-embedding-artifact.v1'
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
os.system(f"{sys.executable} -m pip install --no-deps --disable-pip-version-check {wheel}") == 0 or (_ for _ in ()).throw(RuntimeError('exact wheel installation failed'))

In [ ]:
PRIMARY_SOURCE = '"""Primary runtime source for exact-revision multilingual E5 encoding."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom datetime import UTC, datetime\nfrom pathlib import Path\nfrom uuid import UUID\n\nimport torch\nfrom transformers import AutoModel, AutoTokenizer\n\nfrom my_data_hub.embeddings.contracts import EmbeddingJob\nfrom my_data_hub.embeddings.models import E5_MULTILINGUAL_BASE\nfrom my_data_hub.embeddings.worker import EmbeddingWorker\nfrom my_data_hub.hashing import canonical_json_bytes\n\n\nclass E5Encoder:\n    def __init__(self) -> None:\n        model = E5_MULTILINGUAL_BASE\n        self.tokenizer = AutoTokenizer.from_pretrained(model.model_key, revision=model.revision)\n        self.model = AutoModel.from_pretrained(model.model_key, revision=model.revision).eval()\n\n    def encode(self, texts, *, model, max_tokens, pooling, normalize, dense_only):  # type: ignore[no-untyped-def]\n        if model != E5_MULTILINGUAL_BASE or pooling != "attention_mask_mean" or not normalize or not dense_only:\n            raise ValueError("E5 runtime contract mismatch")\n        encoded = self.tokenizer(list(texts), max_length=max_tokens, padding=True, truncation=True, return_tensors="pt")\n        with torch.inference_mode():\n            hidden = self.model(**encoded).last_hidden_state\n        mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n        vectors = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)\n        return torch.nn.functional.normalize(vectors, p=2, dim=1).cpu().tolist()\n\n\ndef main() -> int:\n    payload = json.loads(Path(os.environ["MY_DATA_HUB_EMBEDDING_JOBS"]).read_text())\n    jobs = tuple(EmbeddingJob.model_validate(item) for item in payload["jobs"])\n    now = datetime.now(UTC)\n    result = EmbeddingWorker(model=E5_MULTILINGUAL_BASE, encoder=E5Encoder()).run(\n        run_id=UUID(os.environ["MY_DATA_HUB_RUN_ID"]), jobs=jobs, started_at=now, completed_at=datetime.now(UTC)\n    )\n    Path("/kaggle/working/embedding-result.json").write_bytes(canonical_json_bytes(result.model_dump(mode="json")))\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())